# 00. Dataset Construction and Storage — GSE225845

This notebook builds the methylation matrix for the tumor subset (GSE225845).  
It performs a full ingestion pipeline from the raw normalized TXT file to efficient Parquet (LZ4) storage, ensuring schema consistency and compatibility with the other GEO cohorts used in the thesis.

**Source: GEO accession GSE225845, platform Illumina HumanMethylation450 BeadChip**

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-preparation-GSE225845                   ║
# ║ Description:  Construction of the core methylation matrix        ║
# ║               (GSE225845) — Parquet storage                      ║
# ║ Dataset(s):   GSE225845                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 10-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [2]:
!pip -q install polars pyarrow


In [3]:
import polars as pl
import os, gc, subprocess, csv
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


## 1. Read TXT → write Parquet (LZ4)

In [ ]:
# READ TXT (streaming) -> WRITE PARQUET LZ4
# 1. Input / output paths
print("1. Input / output paths - Start")
INPUT_TXT = "/kaggle/input/gse225845-tumors-normalized-betas-txt/GSE225845_tumors_normalized_betas.txt"
OUT_PAR = "/kaggle/working/GSE225845_tumor_lz4.parquet"

assert os.path.exists(INPUT_TXT), f"Input file not found: {INPUT_TXT}"
print("1. Input / output paths - End")

# 2. Columns that must stay as strings
print("2. Columns that must stay as strings - Start")
STRING_COLS = ["accession_num", "basenames"]
print("2. Columns that must stay as strings - End")

# 3. Build a lazy CSV scan
# This does NOT load the whole file into memory.
print("3. Build a lazy CSV scan - Start")
lf = pl.scan_csv( INPUT_TXT, 
                 separator="\t", # TXT is tab-separated
                 has_header=True,
                 infer_schema_length=1000, # look at first N rows to guess types
                 null_values=["NA", "NaN", ""] )
print("3. Build a lazy CSV scan - End")

# 4. Cast dtypes lazily:
# - accession_num, basenames -> Utf8 (strings)
# - all other columns -> Float32 (beta values)
print("4. Cast dtypes lazily - Start")
present_string_cols = [c for c in STRING_COLS if c in lf.columns]

exprs = []

# Keep string columns as Utf8
for col in present_string_cols: exprs.append(pl.col(col).cast(pl.Utf8))

# All remaining columns (CpG probes) as Float32
exprs.append( pl.all() .exclude(present_string_cols) .cast(pl.Float32) )

lf = lf.with_columns(exprs)
print("4. Cast dtypes lazily - End")

# 5. Write to Parquet (streaming sink)
# This will stream the data in chunks and never hold the full table in RAM.
print("5. Write to Parquet - Start")
lf.sink_parquet( OUT_PAR, compression="lz4", statistics=True )

print(f"✅ Parquet saved to: {OUT_PAR}")
print("5. Write to Parquet - End")

# 6. Quick sanity check (now we can read the Parquet normally)
print("6. Quick sanity check - Start")
par = pl.read_parquet(OUT_PAR)
print("Parquet shape:", par.shape)

if present_string_cols:
    print(par.select(present_string_cols).head())

print("Parquet dtypes (first 5):", par.dtypes[:5])
print("6. Quick sanity check - End")


## 2. Read TXT → write Parquet (LZ4) for Tumor

In [ ]:
# BUILD ONE PARQUET FILE (LZ4), NO TRANSPOSE, FOR GSE225845 TUMORS
# OPTIMIZATION 5.0 (Sample Count Correction):
# 1. Updates NUM_SAMPLES to 224 to resolve the AssertionError.
# 2. Retains all speed optimizations: no memmap, no wc -l,
#    header read with standard csv module, dtype=float32, names=all_cols.

INPUT_PATH     = "/kaggle/input/gse225845-tumors-normalized-betas-txt/GSE225845_tumors_normalized_betas.txt"
OUTPUT_FILE    = "/kaggle/working/GSE225845_tumors_lz4_float32.parquet"

META_COLS      = ["accession_num", "basenames"]  # metadata cols as in TXT
FLOAT_DTYPE    = np.float32                     # Dtype for the values
NA_STRINGS     = ["NA", "NaN", "nan", "", "null", "NULL"]
ROW_BLOCK_SIZE = 1024  # Number of rows per chunk (large for efficient I/O)
NUM_SAMPLES    = 185

print("STEP 1 - imports and config ready")

def ensure_parent(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)

# 1) Schema Discovery (FAST with the 'csv' module)
assert os.path.exists(INPUT_PATH), f"Input file not found: {INPUT_PATH}"
ensure_parent(OUTPUT_FILE)

print("STEP 2 - reading header with standard CSV module (optimal speed for wide files)")
all_cols: List[str] = []
try:
    with open(INPUT_PATH, 'r', newline='', encoding='utf-8') as f:
        # Use the standard reader to parse the first row quickly
        reader = csv.reader(f, delimiter='\t')
        all_cols = next(reader)
except Exception as e:
    print(f"ERROR: Could not read header from file: {e}")
    # If it fails, we use a fallback: read only the first row with pandas
    try:
        hdr = pd.read_csv(INPUT_PATH, sep="\t", nrows=0)
        all_cols = hdr.columns.tolist()
    except Exception as e_fallback:
        raise Exception(f"FATAL: Both CSV reader and Pandas failed to read the header: {e_fallback}")


print(f"  Total columns: {len(all_cols):,}")

# Metadata and CpG Columns
meta_cols = [c for c in META_COLS if c in all_cols]
cpg_cols  = [c for c in all_cols if c not in meta_cols]

print(f"  Metadata columns: {meta_cols}")
print(f"  CpG columns: {len(cpg_cols):,}")

# I USE THE CONSTANT FOR ROWS
num_samples = NUM_SAMPLES
num_cpgs    = len(cpg_cols)
print(f"[INFO] #samples (rows) = {num_samples:,}")
print(f"[INFO] #CpGs    (cols) = {num_cpgs:,}")

# 2) Define Pyarrow schema and the `dtype` map for Pandas
print("STEP 3 - Building Arrow schema and Pandas dtype map")

# 1. Dtype map for PANDAS (to read float32 DIRECTLY)
dtype_map: Dict[str, any] = {c: "string" for c in meta_cols}
dtype_map.update({c: FLOAT_DTYPE for c in cpg_cols})

# 2. Schema for PYARROW (for the ParquetWriter)
pa_fields = []
for col in meta_cols:
    pa_fields.append(pa.field(col, pa.string()))
for col in cpg_cols:
    pa_fields.append(pa.field(col, pa.float32()))

pa_schema = pa.schema(pa_fields)
print(f"  Schema definition complete.")

# 3) Initialize ParquetWriter
print("STEP 4 - Initializing ParquetWriter")
writer = pq.ParquetWriter(
    OUTPUT_FILE,
    pa_schema,
    compression="lz4",
    use_dictionary=False,
    data_page_size=1 << 20, # 1MB
    write_statistics=True,
)

print(f"STEP 4 - ParquetWriter initialized for: {OUTPUT_FILE}")

# 4) Read TXT in chunks (Skip the header), convert and write to Parquet
print("STEP 5 - Start streaming TXT chunks directly to Parquet")

# The 'skiprows=[0]' parameter skips the header row already read
# The 'names=all_cols' parameter forces the assignment of column names
reader = pd.read_csv(
    INPUT_PATH,
    sep="\t",
    skiprows=[0],     # <-- Skip header row
    names=all_cols,   # <-- Assign correct names
    dtype=dtype_map,
    chunksize=ROW_BLOCK_SIZE,
    na_values=NA_STRINGS,
    keep_default_na=True,
    low_memory=False,
)

written_rows = 0
chunk_idx = 0
try:
    for chunk in reader:
        chunk_idx += 1
        n_rows_chunk = len(chunk)
        
        # Converts the pandas chunk (row-major) to an Arrow table (column-major)
        table_chunk = pa.Table.from_pandas(
            chunk, 
            schema=pa_schema, 
            preserve_index=False
        )
        
        # Writes the chunk to the parquet file
        writer.write_table(table_chunk)
        
        written_rows += n_rows_chunk

        del chunk, table_chunk
        if chunk_idx % 5 == 0:
            gc.collect()
            print(f"  - processed chunk {chunk_idx}, samples written: {written_rows:,}/{num_samples:,}")

finally:
    # 5) Close the writer
    if writer:
        writer.close()
    del reader
    gc.collect()

print("STEP 6 - Finished writing chunks, closing writer.")
assert written_rows == num_samples, f"Written rows {written_rows} != expected {num_samples}"
print(f"[DONE] Wrote single Parquet -> {OUTPUT_FILE}")
print("STEP 7 - All done.")


## 3. Parquet merger: concatenates multiple parquet files (normal, adjacent, tumor)

In [ ]:
# PARQUET MERGER: CONCATENATES MULTIPLE PARQUET FILES INTO A SINGLE FILE
# Files are concatenated vertically (adding rows/samples).
# Adds a 'label' column to identify the sample type.

# List of tuples: (Parquet file path, label value)
FILES_WITH_LABELS: List[Tuple[str, int]] = [
    ("/kaggle/input/gse225845-normal-lz4-parquet/GSE225845_normal_lz4.parquet", 0),  # 0 = normal
    ("/kaggle/input/gse225845-adj-lz4-parquet/GSE225845_adj-lz4.parquet", 1),       # 1 = normal-adjacent
    ("/kaggle/input/gse225845-tumors/GSE225845_tumors_lz4_float32.parquet", 2),  # 2 = breast cancer
]

OUTPUT_PATH = "/kaggle/working/GSE225845_all_samples_with_labels.parquet"

# Columns to keep as strings (accession_num, basenames)
# Ensure 'label' is the first column after metadata for convention.
METADATA_COLS_TO_KEEP = ["accession_num", "basenames"] 
LABEL_COL_NAME = "label"

# Build lazy frames with labels
print("STEP 1: Building Lazy Frames and adding 'label' column...")

lfs = []
for path, lab in FILES_WITH_LABELS:
    # 1. Check if the file exists before proceeding
    if not Path(path).exists():
        print(f"ATTENTION: File not found and skipped: {path}")
        continue
    
    # 2. Read the file in lazy mode
    lf = (
        pl.scan_parquet(path)  # lazy, no full load in RAM
        .with_columns(
            pl.lit(lab)
              .cast(pl.Int8)    # small integer, saves space
              .alias(LABEL_COL_NAME)
        )
    )
    
    # 3. Reorganize columns: metadata + label + CpG
    # This reorganization ensures the schema is uniform before concatenation.
    col_order = [pl.col(c) for c in METADATA_COLS_TO_KEEP]
    col_order.append(pl.col(LABEL_COL_NAME))
    col_order.append(pl.exclude(METADATA_COLS_TO_KEEP + [LABEL_COL_NAME]))
    
    lf = lf.select(*col_order)
    
    lfs.append(lf)
    print(f"  - Added {path} (Label: {lab})")

if not lfs:
    raise FileNotFoundError("FATAL: No valid Parquet file was found. Cannot concatenate.")
    
# Vertical concatenation: same columns, different samples
print("\nSTEP 2: Concatenating all Lazy Frames (Vertical merge)...")

# Polars handles vertical concatenation of frames with identical schemas.
lf_all = pl.concat(lfs, how="vertical")

print("  Concatenation successful (Lazy mode).")

# Optional quick sanity check (small sample)
print("\nSTEP 3: Performing a quick data summary check...")
try:
    summary_df = (
        lf_all
        .select(
            pl.len().alias("n_samples_total"),
            pl.col(LABEL_COL_NAME).value_counts().sort(LABEL_COL_NAME)
        )
        .collect()
    )
    print(summary_df)
except Exception as e:
    print(f"ATTENTION: Could not perform sanity check (perhaps due to schema): {e}")


# Write single Parquet in streaming mode
print(f"\nSTEP 4: Writing single combined Parquet file to {OUTPUT_PATH}...")

# The use of sink_parquet ensures optimized and low-RAM writing
lf_all.sink_parquet(
    OUTPUT_PATH,
    compression="lz4",
    statistics=True,
    # The row_group_size parameter can be useful for subsequent analysis, but we use Polars' default
)

print(f"\n✅ Written combined Parquet with labels -> {OUTPUT_PATH}")
print("Process completed.")
